In [1]:
# Install Unsloth and compatible dependencies
%pip install -q -U pip
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!nvidia-smi

In [1]:
# Final all-in-one script for Jupyter Notebook
# 1) Set aggressive memory optimization
# 2) clone/repair repo
# 3) safely patch PPO config
# 4) train model (with real-time output)
# 5) run benchmarks
# 6) print summary

import os
import sys
import json
import subprocess
from pathlib import Path

# -----------------------------
# MEMORY OPTIMIZATION (CRITICAL FOR L4/T4)
# -----------------------------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# -----------------------------
# HELPER: STREAM OUTPUT TO JUPYTER
# -----------------------------
def run_and_stream(cmd, cwd):
    """Runs a command and streams the output to Jupyter in real-time."""
    process = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1 
    )
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush() 
        
    process.wait()
    if process.returncode != 0:
        print(f"\n❌ Process failed with exit code {process.returncode}")
        sys.exit(process.returncode)

# -----------------------------
# CONFIG
# -----------------------------
REPO_URL = "https://github.com/akhi499/Scaler-Final.git"
BRANCH = "main"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Trying 3B one last time with strict memory limits!
EPISODES = 50 
EVAL_EPISODES = 20
TASK_EVAL_EPISODES = 20
SEED = 42

# -----------------------------
# DIRECTORY SETUP
# -----------------------------
BASE = Path.cwd()
CLONE_DIR = BASE / "zombieshield_env"
OUTPUT_DIR = CLONE_DIR / "training_outputs_qwen"

# -----------------------------
# CLONE / REPAIR REPO
# -----------------------------
if not CLONE_DIR.exists():
    print(f"\nCloning repository into {CLONE_DIR}...")
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(CLONE_DIR)], check=True)
else:
    print(f"\nRepository exists. Repairing any broken files and fetching updates...")
    # This specifically fixes the SyntaxError by discarding our broken Unsloth modifications
    subprocess.run(["git", "-C", str(CLONE_DIR), "checkout", "--", "training/train_trl.py"], check=True)
    subprocess.run(["git", "-C", str(CLONE_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(CLONE_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "origin", BRANCH], check=True)

# -----------------------------
# PATCH train_trl.py (SAFE MODE)
# -----------------------------
print("\nApplying safe batch-size patch to train_trl.py...")
train_file = CLONE_DIR / "training" / "train_trl.py"
src = train_file.read_text(encoding="utf-8")

# Highly targeted string replacement to avoid breaking indentation
if "gradient_accumulation_steps=4)" in src and "batch_size=4" not in src:
    src = src.replace(
        "gradient_accumulation_steps=4)", 
        "batch_size=4, mini_batch_size=1, gradient_accumulation_steps=4)"
    )
    train_file.write_text(src, encoding="utf-8")
    print("✅ Safely patched PPOConfig batch math down to 4 for VRAM savings!")

# Verify syntax
try:
    compile(train_file.read_text(encoding="utf-8"), str(train_file), "exec")
except SyntaxError as e:
    print(f"⚠️ Syntax Error: {e}")
    sys.exit(1)

# -----------------------------
# TRAIN
# -----------------------------
train_cmd = [
    sys.executable, "training/train_trl.py",
    "--model-name", MODEL_NAME,
    "--episodes", str(EPISODES),
    "--eval-episodes", str(EVAL_EPISODES),
    "--log-every", "1", 
    "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
]

print("\n🚀 Running training (Real-time output enabled):\n", " ".join(train_cmd))
print("-" * 50)
run_and_stream(train_cmd, cwd=str(CLONE_DIR))
print("-" * 50)

# -----------------------------
# STANDALONE DIFFICULTY BENCHMARK
# -----------------------------
eval_cmd = [
    sys.executable, "training/evaluate_tasks.py",
    "--model-name", MODEL_NAME,
    "--trained-model-dir", str(OUTPUT_DIR / "trained_model"),
    "--episodes", str(TASK_EVAL_EPISODES),
    "--seed", str(SEED),
    "--output", str(OUTPUT_DIR / "task_benchmark_summary.json"),
]

print("\n📊 Running task benchmark (Real-time output enabled):\n", " ".join(eval_cmd))
print("-" * 50)
run_and_stream(eval_cmd, cwd=str(CLONE_DIR))
print("-" * 50)

# -----------------------------
# PRINT RESULTS
# -----------------------------
summary_path = OUTPUT_DIR / "training_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("\n=== Improvement ===")
    print(json.dumps(summary.get("improvement", {}), indent=2))
else:
    print("\n⚠️ No training summary found at:", summary_path)

Already on 'main'
From https://github.com/akhi499/Scaler-Final
 * branch            main       -> FETCH_HEAD



Repository exists. Repairing any broken files and fetching updates...
Your branch is up to date with 'origin/main'.
Already up to date.

Applying safe batch-size patch to train_trl.py...
✅ Safely patched PPOConfig batch math down to 4 for VRAM savings!

🚀 Running training (Real-time output enabled):
 /home/user/miniconda/bin/python training/train_trl.py --model-name Qwen/Qwen2.5-3B-Instruct --episodes 50 --eval-episodes 20 --log-every 1 --seed 42 --output-dir /data/zombieshield_env/training_outputs_qwen
--------------------------------------------------
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.53s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.51s/it]
/home/user/miniconda/lib/python3.10/site-packages/trl/trainer/ppo_trainer.py:262: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before tr

KeyboardInterrupt: 

In [3]:
# 🚨 EMERGENCY EVALUATION CELL (FIXED IMPORTS) 🚨
import sys
import json
from pathlib import Path

# 1. Force Jupyter to see your local files so we can import from them
sys.path.append(str(Path.cwd()))
sys.path.append(str(Path.cwd() / "zombieshield_env"))

# 2. Import all the missing classes and functions directly from your script!
try:
    from train_trl import (
        TRLPPOPolicy, RandomPolicy, HeuristicPolicy, 
        evaluate_policy, evaluate_across_tasks, 
        _save_baseline_comparison, _save_task_difficulty_comparison
    )
except ModuleNotFoundError:
    # Fallback just in case your script is tucked inside a 'training' folder
    from training.train_trl import (
        TRLPPOPolicy, RandomPolicy, HeuristicPolicy, 
        evaluate_policy, evaluate_across_tasks, 
        _save_baseline_comparison, _save_task_difficulty_comparison
    )

# 3. Configuration
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct" 
OUTPUT_DIR = Path("zombieshield_env/training_outputs_qwen")
BEST_MODEL_PATH = OUTPUT_DIR / "best_trl_model"

print(f"Loading the Best Checkpoint from {BEST_MODEL_PATH}...")

# Load the policy
best_policy = TRLPPOPolicy(model_name=BASE_MODEL_NAME, load_from=str(BEST_MODEL_PATH))

# ---------------------------------------------------------
# RAPID BASELINE COMPARISON (3 Episodes instead of 12)
# ---------------------------------------------------------
print("Running Rapid Baseline Comparison (Generating 3 episodes per agent)...")
env_kwargs = {"min_apis": 10, "max_apis": 30}

random_eval = evaluate_policy(RandomPolicy(42), episodes=3, env_kwargs=env_kwargs, seed_base=100)
heuristic_eval = evaluate_policy(HeuristicPolicy(42), episodes=3, env_kwargs=env_kwargs, seed_base=200)
posttrain_eval = evaluate_policy(best_policy, episodes=3, env_kwargs=env_kwargs, seed_base=300)

# ---------------------------------------------------------
# RAPID DIFFICULTY BENCHMARK (2 Episodes per task instead of 10)
# ---------------------------------------------------------
print("Running Rapid Difficulty Benchmark (Generating 2 episodes per task)...")

random_tasks = evaluate_across_tasks(RandomPolicy(42), episodes=2, seed_base=1000)
heuristic_tasks = evaluate_across_tasks(HeuristicPolicy(42), episodes=2, seed_base=2000)
posttrain_tasks = evaluate_across_tasks(best_policy, episodes=2, seed_base=3000)

# ---------------------------------------------------------
# GENERATE THE CHARTS
# ---------------------------------------------------------
print("Generating final PNG charts...")

_save_baseline_comparison(
    OUTPUT_DIR, random_eval, heuristic_eval, random_eval, posttrain_eval
)

_save_task_difficulty_comparison(
    OUTPUT_DIR, random_tasks, heuristic_tasks, random_tasks, posttrain_tasks
)

# Save a mini-summary JSON for the judges
emergency_summary = {
    "status": "emergency_rapid_eval",
    "agent_posttrain": posttrain_eval,
    "difficulty_eval": posttrain_tasks
}
(OUTPUT_DIR / "emergency_summary.json").write_text(json.dumps(emergency_summary, indent=2))

print(f"✅ DONE! Rapid evaluation complete. Check {OUTPUT_DIR} for your charts!")

Loading the Best Checkpoint from zombieshield_env/training_outputs_qwen/best_trl_model...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/user/miniconda/lib/python3.10/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/user/miniconda/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/user/miniconda/lib/python3.10/site-packages/trl/trainer/ppo_trainer.py:262: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before training.
  warnings.warn(
/home/user/miniconda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=T

Running Rapid Baseline Comparison (Generating 3 episodes per agent)...
Running Rapid Difficulty Benchmark (Generating 2 episodes per task)...
Generating final PNG charts...
✅ DONE! Rapid evaluation complete. Check zombieshield_env/training_outputs_qwen for your charts!


In [4]:
import os
from pathlib import Path
from huggingface_hub import HfApi

# ---------------------------------------------------------
# ---- CONFIG ----
# ---------------------------------------------------------
# Set your token here directly to avoid Jupyter environment variable issues
HF_TOKEN = os.getenv("HF_TOKEN") # Set in Space Secrets

TARGET_REPO = "akhi499/Zombie-API-Qwen-3B"   # This will be your new MODEL repo
REPO_TYPE = "model"

# Folder produced by your training script
SOURCE_OUTPUT = Path("zombieshield_env/training_outputs_qwen")  

# We upload directly to the root of the model repository
TARGET_PATH_IN_REPO = "." 

# ---------------------------------------------------------
# ---- UPLOAD LOGIC ----
# ---------------------------------------------------------
if HF_TOKEN == "hf_YOUR_ACTUAL_TOKEN_HERE":
    raise ValueError("🚨 Wait! You forgot to paste your actual HF_TOKEN in the code!")

api = HfApi(token=HF_TOKEN)

# 1. Create the model repo automatically if it doesn't exist yet
try:
    api.create_repo(repo_id=TARGET_REPO, repo_type=REPO_TYPE, exist_ok=True)
    print(f"✅ Verified repository {TARGET_REPO} exists.")
except Exception as e:
    print(f"Note on repo creation: {e}")

# 2. Upload the files
print(f"🚀 Uploading files to {TARGET_REPO}... (This might take a few minutes for a 3B model!)")
api.upload_folder(
    folder_path=str(SOURCE_OUTPUT),
    repo_id=TARGET_REPO,
    repo_type=REPO_TYPE,
    path_in_repo=TARGET_PATH_IN_REPO,
    allow_patterns=[
        "*.json",
        "*.png",
        "best_trl_model/**",  # This grabs your actual trained model!
    ],
)

print(f"🎉 SUCCESS! Your model and charts are live at: https://huggingface.co/{TARGET_REPO}")

✅ Verified repository akhi499/Zombie-API-Qwen-3B exists.
🚀 Uploading files to akhi499/Zombie-API-Qwen-3B... (This might take a few minutes for a 3B model!)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🎉 SUCCESS! Your model and charts are live at: https://huggingface.co/akhi499/Zombie-API-Qwen-3B
